In [ ]:
from __future__ import annotations

import os
import re
import math
import json
import random
import shutil
import subprocess
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import r3d_18, R3D_18_Weights
from transformers import AutoModel, AutoTokenizer, AutoFeatureExtractor, AutoProcessor

import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

In [ ]:
DEFAULT_LABELS = [
    "Complaint", "Praise", "Apologise", "Thank", "Criticize", "Care", "Agree",
    "Taunt", "Flaunt", "Oppose", "Joke", "Doubt", "Acknowledge", "Refuse",
    "Warn", "Emphasize", "Inform", "Advise", "Arrange", "Introduce", "Comfort",
    "Leave", "Prevent", "Greet", "Ask for help", "Ask for opinions", "Confirm",
    "Explain", "Invite", "Plan", "Other",
]

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpg", ".mpeg", ".m4v", ".wmv"}
TARGET_AUDIO_SR = 16000

In [ ]:
@dataclass
class TrainConfig:
    excel_path: str
    video_dir: str
    save_dir: str = "/content/visk_outputs"
    text_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    audio_model_name: str = "microsoft/wavlm-base-plus"
    num_frames: int = 8
    img_size: int = 112
    batch_size: int = 1
    epochs: int = 10
    lr: float = 2e-5
    weight_decay: float = 0.01
    seed: int = 42
    freeze_text_encoder: bool = True
    freeze_audio_encoder: bool = True
    pretrained_video_backbone: bool = True
    iterations: int = 2
    sigma_init_decay: float = 0.05
    perturb_dropout: float = 0.1
    dropout: float = 0.2
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    num_workers: int = 0
    max_rows: Optional[int] = None
    use_start_end_times: bool = True
    max_audio_seconds: float = 12.0

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def normalize_label(label: str) -> str:
    return re.sub(r"\s+", " ", str(label).strip()).lower()


def build_label_mapping(df_labels: Sequence[str], default_labels: Sequence[str] = DEFAULT_LABELS) -> Tuple[Dict[str, int], List[str]]:
    labels = list(default_labels)
    seen = {normalize_label(x) for x in labels}
    for lab in df_labels:
        key = normalize_label(lab)
        if key not in seen:
            labels.append(str(lab).strip())
            seen.add(key)
    label_to_id = {normalize_label(lbl): i for i, lbl in enumerate(labels)}
    return label_to_id, labels

In [ ]:
def find_video_file(video_dir: str, video_id: str) -> Optional[str]:
    if pd.isna(video_id):
        return None
    raw = str(video_id).strip()
    if not raw:
        return None

    if os.path.isabs(raw) and os.path.exists(raw):
        return raw

    candidate = os.path.join(video_dir, raw)
    if os.path.exists(candidate):
        return candidate

    stem = os.path.splitext(raw)[0]
    matches = []
    if os.path.isdir(video_dir):
        for fn in os.listdir(video_dir):
            path = os.path.join(video_dir, fn)
            if not os.path.isfile(path):
                continue
            if os.path.splitext(fn)[1].lower() not in VIDEO_EXTS:
                continue
            base = os.path.splitext(fn)[0]
            if raw == base or stem == base or raw in fn or stem in base:
                matches.append(path)

    if not matches:
        return None
    matches.sort(key=lambda p: (len(os.path.basename(p)), os.path.basename(p)))
    return matches[0]


def safe_mkdir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

In [ ]:
def read_excel_dataset(excel_path: str, video_dir: str, default_labels: Sequence[str]) -> Tuple[pd.DataFrame, List[str]]:
    df = pd.read_excel(excel_path)
    required_cols = {"Video ID", "Hinglish Text", "Hindi Text", "Label", "Start time", "End time"}
    missing = required_cols - set(df.columns)


    df = df.copy()
    df["Label"] = df["Label"].astype(str).str.strip()
    df["video_path"] = df["Video ID"].apply(lambda x: find_video_file(video_dir, x))
    df = df[df["video_path"].notna()].reset_index(drop=True)

    label_to_id, label_list = build_label_mapping(df["Label"].tolist(), default_labels)
    df["label_id"] = df["Label"].apply(lambda x: label_to_id[normalize_label(x)])
    return df, label_list


def make_text_column(df: pd.DataFrame, use_both: bool = True) -> pd.Series:
    hinglish = df["Hinglish Text"].fillna("").astype(str).str.strip()
    hindi = df["Hindi Text"].fillna("").astype(str).str.strip()
    if use_both:
        combined = (hinglish + " [SEP] " + hindi).str.strip()
    else:
        combined = hinglish
    return combined.replace({"": "[UNK]"})

In [ ]:
def time_to_seconds(t) -> Optional[float]:
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return None
    if isinstance(t, (int, float)):
        return float(t)
    s = str(t).strip()
    if not s:
        return None
    parts = s.split(":")
    try:
        if len(parts) == 3:
            hh, mm, ss = parts
            return float(hh) * 3600.0 + float(mm) * 60.0 + float(ss)
        if len(parts) == 2:
            mm, ss = parts
            return float(mm) * 60.0 + float(ss)
        return float(s)
    except Exception:
        return None


def safe_split_dataframe(
    df: pd.DataFrame,
    seed: int,
    train_frac: float,
    val_frac: float,
    test_frac: float,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not math.isclose(train_frac + val_frac + test_frac, 1.0, rel_tol=1e-6, abs_tol=1e-6):
        raise ValueError("train_frac + val_frac + test_frac must equal 1.0")

    if len(df) < 3:
        raise ValueError("Need at least 3 rows after filtering videos.")

    label_counts = df["label_id"].value_counts()
    rare_labels = label_counts[label_counts < 3].index.tolist()
    rare_df = df[df["label_id"].isin(rare_labels)].copy()
    common_df = df[~df["label_id"].isin(rare_labels)].copy()

    train_parts, val_parts, test_parts = [rare_df], [], []

    if len(common_df) == 0:
        shuffled = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n = len(shuffled)
        n_test = int(round(n * test_frac))
        n_val = int(round(n * val_frac))
        n_train = max(n - n_val - n_test, 1)
        return (
            shuffled.iloc[:n_train].reset_index(drop=True),
            shuffled.iloc[n_train:n_train + n_val].reset_index(drop=True),
            shuffled.iloc[n_train + n_val:].reset_index(drop=True),
        )

    if common_df["label_id"].value_counts().min() >= 2 and len(common_df) >= 3:
        temp_ratio = val_frac + test_frac
        train_common, temp_common = train_test_split(
            common_df,
            test_size=temp_ratio,
            random_state=seed,
            stratify=common_df["label_id"],
        )
        rel_test = test_frac / temp_ratio
        # second split: avoid stratify because temp_common can still be tiny
        val_common, test_common = train_test_split(
            temp_common,
            test_size=rel_test,
            random_state=seed,
            shuffle=True,
        )
        train_parts.append(train_common)
        val_parts.append(val_common)
        test_parts.append(test_common)
    else:
        shuffled = common_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n = len(shuffled)
        n_test = int(round(n * test_frac))
        n_val = int(round(n * val_frac))
        n_train = max(n - n_val - n_test, 1)
        train_parts.append(shuffled.iloc[:n_train].copy())
        val_parts.append(shuffled.iloc[n_train:n_train + n_val].copy())
        test_parts.append(shuffled.iloc[n_train + n_val:].copy())

    train_df = pd.concat(train_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_df = pd.concat(val_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True) if val_parts else pd.DataFrame(columns=df.columns)
    test_df = pd.concat(test_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True) if test_parts else pd.DataFrame(columns=df.columns)
    return train_df, val_df, test_df

In [ ]:
def sample_video_clip(
    video_path: str,
    start_sec: Optional[float],
    end_sec: Optional[float],
    num_frames: int = 8,
    img_size: int = 112,
) -> torch.Tensor:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return torch.zeros(num_frames, 3, img_size, img_size, dtype=torch.float32)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 25.0

    if total_frames <= 0:
        cap.release()
        return torch.zeros(num_frames, 3, img_size, img_size, dtype=torch.float32)

    start_idx = 0
    end_idx = max(total_frames - 1, 0)

    if start_sec is not None:
        start_idx = max(int(start_sec * fps), 0)
    if end_sec is not None:
        end_idx = max(int(end_sec * fps), start_idx + 1)

    start_idx = min(start_idx, total_frames - 1)
    end_idx = min(end_idx, total_frames - 1)
    if end_idx <= start_idx:
        start_idx, end_idx = 0, total_frames - 1

    frame_indices = np.linspace(start_idx, end_idx, num_frames).astype(int).tolist()

    preprocess = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])

    frames: List[torch.Tensor] = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            frames.append(torch.zeros(3, img_size, img_size, dtype=torch.float32))
        else:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(preprocess(rgb))

    cap.release()
    return torch.stack(frames, dim=0)


def sanitize_filename(text: str) -> str:
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9._-]+", "_", text)
    return text[:180].strip("_") or "sample"

In [ ]:
def extract_audio_cache(
    video_path: str,
    cache_dir: str,
    video_id: str,
    start_sec: Optional[float],
    end_sec: Optional[float],
    max_audio_seconds: float = 12.0,
) -> Optional[str]:
    safe_mkdir(cache_dir)
    base = sanitize_filename(f"{video_id}_{os.path.splitext(os.path.basename(video_path))[0]}")
    if start_sec is not None and end_sec is not None:
        duration = max(0.1, min(float(end_sec) - float(start_sec), max_audio_seconds))
        tag = f"{start_sec:.2f}_{end_sec:.2f}"
    else:
        duration = max_audio_seconds
        tag = "full"
    out_path = os.path.join(cache_dir, f"{base}_{tag}.wav")
    if os.path.exists(out_path):
        return out_path

    cmd = [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-ss",
        str(max(start_sec or 0.0, 0.0)),
        "-t",
        str(duration),
        "-i",
        video_path,
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(TARGET_AUDIO_SR),
        out_path,
    ]

    try:
        subprocess.run(cmd, check=True)
        if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
            return out_path
    except Exception:
        pass

    if os.path.exists(out_path):
        try:
            os.remove(out_path)
        except Exception:
            pass
    return None


def load_audio_waveform(audio_path: Optional[str], target_sr: int = TARGET_AUDIO_SR) -> np.ndarray:
    if audio_path is None or not os.path.exists(audio_path):
        return np.zeros(target_sr, dtype=np.float32)
    try:
        waveform, sr = torchaudio.load(audio_path)
        waveform = waveform.float()
        if waveform.ndim == 2:
            waveform = waveform.mean(dim=0)
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)
        waveform = waveform.numpy().astype(np.float32)
        if waveform.size == 0:
            return np.zeros(target_sr, dtype=np.float32)
        return waveform
    except Exception:
        return np.zeros(target_sr, dtype=np.float32)

In [ ]:
class HindiHinglishMultimodalDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        config: TrainConfig,
        label2id: Dict[str, int],
        train: bool = False,
    ):
        self.df = df.reset_index(drop=True)
        self.config = config
        self.label2id = label2id
        self.train = train
        self.audio_cache_dir = os.path.join(config.save_dir, "audio_cache")
        safe_mkdir(self.audio_cache_dir)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        video_id = str(row["Video ID"]).strip()
        hindi_text = str(row["Hindi Text"]).strip()
        hinglish_text = str(row["Hinglish Text"]).strip()
        label = str(row["Label"]).strip()
        start_sec = time_to_seconds(row["Start time"]) if self.config.use_start_end_times else None
        end_sec = time_to_seconds(row["End time"]) if self.config.use_start_end_times else None

        video_path = str(row["video_path"])
        video = sample_video_clip(
            video_path=video_path,
            start_sec=start_sec,
            end_sec=end_sec,
            num_frames=self.config.num_frames,
            img_size=self.config.img_size,
        )

        audio_path = extract_audio_cache(
            video_path=video_path,
            cache_dir=self.audio_cache_dir,
            video_id=video_id,
            start_sec=start_sec,
            end_sec=end_sec,
            max_audio_seconds=self.config.max_audio_seconds,
        )

        return {
            "video": video,
            "audio_path": audio_path,
            "hindi_text": hindi_text,
            "hinglish_text": hinglish_text,
            "label": self.label2id[normalize_label(label)],
            "video_id": video_id,
        }


def collate_fn(batch: List[Dict]) -> Dict[str, object]:
    return {
        "videos": torch.stack([x["video"] for x in batch], dim=0),
        "audio_paths": [x["audio_path"] for x in batch],
        "hindi_texts": [x["hindi_text"] for x in batch],
        "hinglish_texts": [x["hinglish_text"] for x in batch],
        "labels": torch.tensor([x["label"] for x in batch], dtype=torch.long),
        "video_ids": [x["video_id"] for x in batch],
    }

Encoders


In [ ]:
class MeanPoolTextEncoder(nn.Module):
    def __init__(self, model_name: str, freeze: bool = True):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        self.out_dim = self.encoder.config.hidden_size
        self.freeze = freeze
        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False

    @staticmethod
    def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
        summed = torch.sum(last_hidden_state * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-6)
        return summed / counts

    def forward(self, texts: List[str], device: torch.device) -> torch.Tensor:
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)

        if self.freeze:
            with torch.no_grad():
                outputs = self.encoder(**encoded)
        else:
            outputs = self.encoder(**encoded)

        pooled = self.mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
        return pooled

In [ ]:
class WavLMEmbeddingEncoder(nn.Module):
    def __init__(self, model_name: str, freeze: bool = True, target_sr: int = TARGET_AUDIO_SR):
        super().__init__()
        self.feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        self.out_dim = self.encoder.config.hidden_size
        self.freeze = freeze
        self.target_sr = target_sr

        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False

    def forward(self, audio_paths: List[Optional[str]], device: torch.device) -> torch.Tensor:
        waveforms = [load_audio_waveform(path, self.target_sr) for path in audio_paths]

        inputs = self.feature_extractor(
            waveforms,
            sampling_rate=self.target_sr,
            return_tensors="pt",
            padding=True,
        )
        inputs = {k: v.to(device) for k, v in inputs.items() if torch.is_tensor(v)}

        if self.freeze:
            with torch.no_grad():
                outputs = self.encoder(**inputs)
        else:
            outputs = self.encoder(**inputs)

        # safest pooling for WavLM hidden states
        pooled = outputs.last_hidden_state.mean(dim=1)
        return pooled

In [ ]:
class R3D18Backbone(nn.Module):

    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = R3D_18_Weights.DEFAULT if pretrained else None
        model = r3d_18(weights=weights)
        self.stem = model.stem
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.out_dim = 512

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # B,T,C,H,W -> B,C,T,H,W
        x = x.permute(0, 2, 1, 3, 4).contiguous().float()
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x


class PerturbationNet(nn.Module):
    def __init__(self, in_channels: int, channels_1: int = 256, channels_2: int = 128, dropout: float = 0.1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, channels_1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(channels_1, channels_2, kernel_size=3, padding=1)
        self.conv3 = nn.Conv3d(channels_2, 1, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout3d(dropout)
        self.softplus = nn.Softplus()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: B, I, C, T, H, W
        b, i, c, t, h, w = x.shape
        x = x.view(b * i, c, t, h, w)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.drop(x)
        x = self.softplus(self.conv3(x))
        return x.view(b, i, 1, t, h, w)


class VisKVideoEncoder(nn.Module):
    def __init__(
        self,
        num_classes: int,
        iterations: int = 2,
        sigma_init_decay: float = 0.05,
        pretrained_backbone: bool = True,
        perturb_dropout: float = 0.1,
    ):
        super().__init__()
        self.backbone = R3D18Backbone(pretrained=pretrained_backbone)
        self.iterations = iterations
        self.sigma_init_decay = sigma_init_decay
        self.backbone_dim = self.backbone.out_dim

        self.perturb_net = PerturbationNet(
            in_channels=self.backbone_dim,
            channels_1=max(128, self.backbone_dim // 2),
            channels_2=max(64, self.backbone_dim // 4),
            dropout=perturb_dropout,
        )

        self.local_cls = nn.Linear(self.backbone_dim, num_classes)
        self.semantic_proj = nn.Linear(self.backbone_dim, 256)

    def forward(self, videos: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feature_map = self.backbone(videos)  # B,C,T,H,W
        b, c, t, h, w = feature_map.shape

        # tokens for local quantification
        f = feature_map.flatten(2).transpose(1, 2).contiguous()  # B,N,C
        y = self.local_cls(f)  # B,N,classes

        # repeat features for perturbation trials
        x_rep = feature_map.unsqueeze(1).expand(b, self.iterations, c, t, h, w).contiguous()
        unit_noise = torch.randn_like(x_rep)
        sigma = self.perturb_net(x_rep)  # B,I,1,T,H,W
        perturb = sigma * unit_noise

        # scale perturbation like the paper
        eps = 1e-6
        p_m = perturb.norm(p=2, dim=2)  # B,I,T,H,W
        i_m = x_rep.norm(p=2, dim=2)     # B,I,T,H,W
        scale = (i_m / (p_m + eps)) * self.sigma_init_decay
        scale = torch.min(scale, torch.ones_like(scale))
        x_pert = x_rep + perturb * scale.unsqueeze(2)

        # perturbed tokens
        f_pert = x_pert.flatten(3).permute(0, 1, 3, 2).contiguous()  # B,I,N,C
        y_pert = self.local_cls(f_pert)  # B,I,N,classes

        delta_f = torch.abs(f_pert - f.unsqueeze(1)).mean(dim=-1)  # B,I,N
        delta_y = F.mse_loss(y_pert, y.unsqueeze(1), reduction="none").sum(dim=-1)  # B,I,N
        kappa = (delta_y / (delta_f + eps)).mean(dim=1).detach()  # B,N

        # weighted semantic representation
        weighted = f * kappa.unsqueeze(-1)  # B,N,C
        semantic = weighted.mean(dim=1)  # B,C
        semantic = self.semantic_proj(semantic)  # B,256
        return semantic, kappa

In [ ]:
class MultimodalVisKModel(nn.Module):
    def __init__(
        self,
        num_classes: int,
        text_model_name: str,
        audio_model_name: str,
        freeze_text_encoder: bool = True,
        freeze_audio_encoder: bool = True,
        pretrained_video_backbone: bool = True,
        fusion_dim: int = 256,
        iterations: int = 2,
        sigma_init_decay: float = 0.05,
        perturb_dropout: float = 0.1,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.text_encoder = MeanPoolTextEncoder(text_model_name, freeze=freeze_text_encoder)
        self.audio_encoder = WavLMEmbeddingEncoder(audio_model_name, freeze=freeze_audio_encoder)
        self.video_encoder = VisKVideoEncoder(
            num_classes=num_classes,
            iterations=iterations,
            sigma_init_decay=sigma_init_decay,
            pretrained_backbone=pretrained_video_backbone,
            perturb_dropout=perturb_dropout,
        )

        text_dim = self.text_encoder.out_dim
        audio_dim = self.audio_encoder.out_dim

        self.video_proj = nn.Linear(256, fusion_dim)
        self.audio_proj = nn.Linear(audio_dim, fusion_dim)
        self.hindi_proj = nn.Linear(text_dim, fusion_dim)
        self.hinglish_proj = nn.Linear(text_dim, fusion_dim)
        self.text_pair_proj = nn.Sequential(
            nn.Linear(text_dim * 4, fusion_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim),
        )

        self.modality_type = nn.Parameter(torch.zeros(5, fusion_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=fusion_dim,
            nhead=4,
            dim_feedforward=fusion_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.classifier = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes),
        )

    def forward(
        self,
        videos: torch.Tensor,
        hindi_texts: List[str],
        hinglish_texts: List[str],
        audio_paths: List[Optional[str]],
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        device = videos.device

        hindi_emb = self.text_encoder(hindi_texts, device)
        hinglish_emb = self.text_encoder(hinglish_texts, device)
        audio_emb = self.audio_encoder(audio_paths, device)

        text_pair = self.text_pair_proj(torch.cat([
            hindi_emb,
            hinglish_emb,
            torch.abs(hindi_emb - hinglish_emb),
            hindi_emb * hinglish_emb,
        ], dim=-1))

        video_semantic, kappa = self.video_encoder(videos)

        v = self.video_proj(video_semantic)
        a = self.audio_proj(audio_emb)
        h1 = self.hindi_proj(hindi_emb)
        h2 = self.hinglish_proj(hinglish_emb)
        t = text_pair

        tokens = torch.stack([v, a, h1, h2, t], dim=1)  # B,5,D
        tokens = tokens + self.modality_type.unsqueeze(0)

        fused = self.fusion_transformer(tokens)
        pooled = fused.mean(dim=1)
        logits = self.classifier(pooled)

        return logits, {
            "video_semantic": video_semantic,
            "kappa": kappa,
            "audio_emb": audio_emb,
            "hindi_emb": hindi_emb,
            "hinglish_emb": hinglish_emb,
            "text_pair": text_pair,
        }

In [ ]:
def compute_metrics(y_true: Sequence[int], y_pred: Sequence[int], num_classes: int) -> Dict[str, float]:
    labels_idx = list(range(num_classes))
    return {
        "ACC": accuracy_score(y_true, y_pred),
        "WF1": f1_score(y_true, y_pred, average="weighted", labels=labels_idx, zero_division=0),
    }


class EarlyStopping:
    def __init__(self, patience: int = 5):
        self.patience = patience
        self.best_score = -float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, score: float, model: nn.Module) -> bool:
        if score > self.best_score:
            self.best_score = score
            self.counter = 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.counter += 1
        return self.counter >= self.patience

In [ ]:
def class_weights_from_train(train_labels: List[int], num_classes: int, device: torch.device) -> torch.Tensor:
    counts = np.bincount(np.array(train_labels, dtype=np.int64), minlength=num_classes).astype(np.float32)
    weights = np.ones(num_classes, dtype=np.float32)
    present = counts > 0
    if present.any():
        inv = counts[present].sum() / (present.sum() * counts[present])
        inv = inv / inv.mean()
        weights[present] = inv
    return torch.tensor(weights, dtype=torch.float32, device=device)

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, dataloader: DataLoader, device: torch.device, num_classes: int, id2label: Dict[int, str]):
    model.eval()
    all_preds, all_true = [], []

    for batch in dataloader:
        videos = batch["videos"].to(device)
        labels = batch["labels"].to(device)
        logits, _ = model(videos, batch["hindi_texts"], batch["hinglish_texts"], batch["audio_paths"])
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_true.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_true, all_preds, num_classes=num_classes)
    report = classification_report(
        all_true,
        all_preds,
        labels=list(range(num_classes)),
        target_names=[id2label[i] for i in range(num_classes)],
        digits=4,
        zero_division=0,
    )
    cm = confusion_matrix(all_true, all_preds, labels=list(range(num_classes)))
    return {
        **metrics,
        "report": report,
        "cm": cm,
        "y_true": all_true,
        "y_pred": all_preds,
    }

In [ ]:
def display_correct_predictions(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    id2label: Dict[int, str],
    num_examples: int = 6,
):
    model.eval()
    picked = []

    with torch.no_grad():
        for batch in dataloader:
            videos = batch["videos"].to(device)
            labels = batch["labels"].to(device)
            logits, _ = model(videos, batch["hindi_texts"], batch["hinglish_texts"], batch["audio_paths"])
            preds = logits.argmax(dim=1)

            for i in range(len(labels)):
                if preds[i].item() == labels[i].item():
                    picked.append({
                        "video": batch["videos"][i].cpu(),
                        "text_hindi": batch["hindi_texts"][i],
                        "text_hinglish": batch["hinglish_texts"][i],
                        "true": id2label[labels[i].item()],
                        "pred": id2label[preds[i].item()],
                    })
                    if len(picked) >= num_examples:
                        break
            if len(picked) >= num_examples:
                break

    if not picked:
        print("No correctly classified examples found for demo.")
        return

    print(f"\nShowing {len(picked)} correctly classified test examples:\n")
    for i, ex in enumerate(picked, 1):
        print(f"Example {i}")
        print(f"True label : {ex['true']}")
        print(f"Pred label : {ex['pred']}")
        print(f"Hindi      : {ex['text_hindi']}")
        print(f"Hinglish   : {ex['text_hinglish']}")
        video = ex["video"]
        mid = video.shape[0] // 2
        frame = video[mid].permute(1, 2, 0).numpy()
        frame = np.clip(frame, 0, 1)
        plt.figure(figsize=(4, 4))
        plt.imshow(frame)
        plt.axis("off")
        plt.show()
        print("-" * 60)